# Test Model


In [16]:
import os
import time
import subprocess
import requests
import base64
import ollama

os.system("pkill -9 ollama")
os.system("nohup ollama serve > /dev/null 2>&1 &")

for _ in range(30):
    try:
        if requests.get("http://127.0.0.1:11434").status_code == 200:
            break
    except:
        time.sleep(1)

os.system("ollama pull llava > /dev/null 2>&1")
os.system("wget -q -O test.jpg 'https://images.unsplash.com/photo-1514888286974-6c03e2ca1dba?w=400'")

with open("test.jpg", "rb") as f:
    img = base64.b64encode(f.read()).decode("utf-8")

response = ollama.chat(
    model='llava',
    messages=[{'role': 'user', 'content': 'Describe this image.', 'images': [img]}],
    stream=True
)

for chunk in response:
    print(chunk['message']['content'], end="", flush=True)

 The image features a gray and white cat sitting on a ledge. The cat has two ears, four legs, and one tail. It is looking directly at the camera with its eyes. The background of the photo is a plain green wall. There are no texts or other objects in the image. 

# Exercise 1

In [19]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import base64
import ollama
import operator
from typing import TypedDict, List, Annotated
from langgraph.graph import StateGraph, END
from langchain_core.messages import HumanMessage, AIMessage, BaseMessage

# LangGraph
class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], operator.add]
    current_image: str

def call_vlm(state: AgentState):
    messages = state['messages']
    current_image = state.get('current_image')
    last_user_msg = messages[-1].content

    history = ""
    if len(messages) > 1:
        history = "History:\n" + "\n".join(
            [f"{'User' if isinstance(m, HumanMessage) else 'AI'}: {m.content}"
             for m in messages[:-1]]
        )

    prompt = f"{history}\n\nQuestion: {last_user_msg}" if history else last_user_msg

    try:
        response = ollama.chat(
            model='llava',
            messages=[{
                'role': 'user',
                'content': prompt,
                'images': [current_image] if current_image else []
            }]
        )
        ai_response = response['message']['content']
    except Exception as e:
        ai_response = f"Error: {str(e)}"

    return {"messages": [AIMessage(content=ai_response)]}

workflow = StateGraph(AgentState)
workflow.add_node("vlm_node", call_vlm)
workflow.set_entry_point("vlm_node")
workflow.add_edge("vlm_node", END)
app = workflow.compile()

# UI
chat_state = {"messages": [], "current_image": None}

title = widgets.HTML("<h2>🖼️ LLaVA Vision Chat</h2>")
upload_btn = widgets.FileUpload(accept='image/*', multiple=False, description='1. Upload Image')
image_out = widgets.Image(width=300)
chat_log = widgets.Output(layout={'border': '1px solid #ccc', 'height': '300px', 'overflow_y': 'auto', 'padding': '10px'})
text_input = widgets.Text(placeholder='2. Ask a question about the image...', layout={'width': '70%'})
send_btn = widgets.Button(description='Send', button_style='primary')
status_label = widgets.Label(value="")

def on_upload(change):
    if upload_btn.value:
        uploaded = upload_btn.value[0] if isinstance(upload_btn.value, (list, tuple)) else list(upload_btn.value.values())[0]
        content = uploaded['content']

        image_out.value = content
        chat_state['current_image'] = base64.b64encode(content).decode('utf-8')

        with chat_log:
            print("---  Image uploaded successfully ---")

upload_btn.observe(on_upload, names='value')

def on_send(b):
    user_msg = text_input.value.strip()
    if not user_msg: return

    text_input.value = ""
    with chat_log:
        print(f" You: {user_msg}")

    status_label.value = " LLaVA is thinking..."

    input_state = {
        "messages": chat_state["messages"] + [HumanMessage(content=user_msg)],
        "current_image": chat_state["current_image"]
    }

    try:
        result = app.invoke(input_state)
        chat_state["messages"] = result["messages"]
        ai_reply = result["messages"][-1].content

        with chat_log:
            print(f" LLaVA: {ai_reply}\n")
    except Exception as e:
        with chat_log:
            print(f" Error: {e}\n")

    status_label.value = ""

send_btn.on_click(on_send)
text_input.on_submit(on_send)

# Display
ui = widgets.VBox([
    title,
    upload_btn,
    image_out,
    chat_log,
    status_label,
    widgets.HBox([text_input, send_btn])
])

display(ui)

# Exercise 2

In [15]:
import cv2
import ollama
import base64
import time

def extract_frames(video_path, seconds_interval=2):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    interval = int(fps * seconds_interval)

    frames_data = []
    frame_num = 0

    print(f"Extracting frames every {seconds_interval} seconds...")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        if frame_num % interval == 0:
            timestamp = frame_num / fps
            # Convert frame to Base64 for Ollama
            _, buffer = cv2.imencode('.jpg', frame)
            img_b64 = base64.b64encode(buffer).decode('utf-8')
            frames_data.append({"time": timestamp, "image": img_b64})
        frame_num += 1

    cap.release()
    return frames_data

def run_analysis(frames):
    print("Starting LLaVA analysis...")
    results = []

    for entry in frames:
        ts = entry["time"]
        time_str = time.strftime('%M:%S', time.gmtime(ts))

        prompt = "Is there a person in this scene? Answer only YES or NO."

        try:
            response = ollama.chat(
                model='llava',
                messages=[{'role': 'user', 'content': prompt, 'images': [entry["image"]]}]
            )
            answer = response['message']['content'].strip().upper()
            is_present = "YES" in answer
            results.append({"time": ts, "time_str": time_str, "person": is_present})
            print(f"[{time_str}] Detection: {'Person present' if is_present else 'Empty'}")
        except Exception as e:
            print(f"Error at {time_str}: {e}")

    return results

def report_events(results):
    print("\n--- Surveillance Report ---")
    person_detected = False

    for res in results:
        # Simple state machine for enter/exit events
        if res["person"] and not person_detected:
            print(f"Event: Person ENTERED at {res['time_str']}")
            person_detected = True
        elif not res["person"] and person_detected:
            print(f"Event: Person EXITED at {res['time_str']}")
            person_detected = False

    if person_detected:
        print("Note: Person was still present at end of video.")

# Main Execution
video_file = "1.mp4"
extracted_frames = extract_frames(video_file)
analysis_results = run_analysis(extracted_frames)
report_events(analysis_results)

Extracting frames every 2 seconds...
Starting LLaVA analysis...
[00:00] Detection: Empty
[00:01] Detection: Empty
[00:03] Detection: Person present
[00:05] Detection: Empty
[00:07] Detection: Empty
[00:09] Detection: Person present
[00:11] Detection: Empty

--- Surveillance Report ---
Event: Person ENTERED at 00:03
Event: Person EXITED at 00:05
Event: Person ENTERED at 00:09
Event: Person EXITED at 00:11
